In [1]:
import kagglehub
path = kagglehub.dataset_download("lakshmi25npathi/imdb-dataset-of-50k-movie-reviews")

Using Colab cache for faster access to the 'imdb-dataset-of-50k-movie-reviews' dataset.


In [2]:
import os
os.listdir(path)

['IMDB Dataset.csv']

In [3]:
import pandas as pd
df=pd.read_csv(path+"/IMDB Dataset.csv")

In [4]:
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [5]:
df.shape

(50000, 2)

In [6]:
df.isnull().sum()

,0
review,0
sentiment,0


## There are no nulls in the dataset

In [7]:
df.duplicated().sum()

np.int64(418)

## There are 418 duplicates found in dataset

In [8]:
df.drop_duplicates(inplace=True)

In [9]:
df.shape

(49582, 2)

In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 49582 entries, 0 to 49999
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   review     49582 non-null  object
 1   sentiment  49582 non-null  object
dtypes: object(2)
memory usage: 1.1+ MB


In [11]:
df['sentiment'].value_counts()

,count
sentiment,
positive,24884
negative,24698


## We can say that data is uniformly distributed.

In [12]:
import re
def cleaner(col):
  col=re.sub('<.*?>','',col)
  col=re.sub('https?://\S+|www\.\S+','',col)
  return re.sub('[^\w\s]','',col)
df['review']=df['review'].apply(cleaner)

<>:4: SyntaxWarning: invalid escape sequence '\S'
<>:5: SyntaxWarning: invalid escape sequence '\w'
<>:4: SyntaxWarning: invalid escape sequence '\S'
<>:5: SyntaxWarning: invalid escape sequence '\w'
/tmp/ipykernel_1487/515907758.py:4: SyntaxWarning: invalid escape sequence '\S'
  col=re.sub('https?://\S+|www\.\S+','',col)
/tmp/ipykernel_1487/515907758.py:5: SyntaxWarning: invalid escape sequence '\w'
  return re.sub('[^\w\s]','',col)


In [13]:
import nltk
nltk.download('stopwords')
nltk.download('punkt_tab')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [14]:
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize,sent_tokenize
words=stopwords.words('english')

In [15]:
def stopword_removal(col):
  L=[]
  tokens=word_tokenize(col)
  for i in tokens:
    if i not in words:
      L.append(i)
  return " ".join(L)
df['review']=df['review'].apply(stopword_removal)

In [16]:
from nltk.stem import SnowballStemmer
sbs=SnowballStemmer('english')
def stemmer(col):
  L=[]
  tokens=word_tokenize(col)
  for i in tokens:
    L.append(sbs.stem(i))
  return " ".join(L)
df['review']=df['review'].apply(stemmer)

In [17]:
df['review'][0]

'one review mention watch 1 oz episod youll hook they right exact happen meth first thing struck oz brutal unflinch scene violenc set right word go trust show faint heart timid this show pull punch regard drug sex violenc it hardcor classic use wordit call oz nicknam given oswald maximum secur state penitentari it focus main emerald citi experiment section prison cell glass front face inward privaci high agenda em citi home manyaryan muslim gangsta latino christian italian irish moreso scuffl death stare dodgi deal shadi agreement never far awayi would say main appeal show due fact goe show wouldnt dare forget pretti pictur paint mainstream audienc forget charm forget romanceoz doesnt mess around the first episod i ever saw struck nasti surreal i couldnt say i readi i watch i develop tast oz got accustom high level graphic violenc not violenc injustic crook guard wholl sold nickel inmat wholl kill order get away well manner middl class inmat turn prison bitch due lack street skill pris

In [18]:
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf=TfidfVectorizer()
vectors=tfidf.fit_transform(df['review'])

In [19]:
from sklearn.preprocessing import LabelEncoder
le=LabelEncoder()
y=le.fit_transform(df['sentiment'])

In [20]:
df.head(4)

,review,sentiment
0,one review mention watch 1 oz episod youll hoo...,positive
1,a wonder littl product the film techniqu unass...,positive
2,i thought wonder way spend time hot summer wee...,positive
3,basic there famili littl boy jake think there ...,negative


In [21]:
y

array([1, 1, 1, ..., 0, 0, 0])

## 1 -> Positive Sentiment
## 0 -> Negative Sentiment

In [22]:
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test=train_test_split(vectors,y,test_size=0.2,random_state=42)

In [23]:
x_train.shape,x_test.shape,y_train.shape,y_test.shape

((39665, 181446), (9917, 181446), (39665,), (9917,))

In [24]:
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import BernoulliNB
from sklearn.metrics import accuracy_score,classification_report

In [25]:
models={
    'Logistic Regression':LogisticRegression(),
    'Bernoulli Naive Bayes':BernoulliNB()
}
accuracy={}
for i in range(len(list(models))):
  model=list(models.values())[i]
  model.fit(x_train,y_train)
  y_train_pred=model.predict(x_train)
  y_test_pred=model.predict(x_test)
  model_train_accuracy=accuracy_score(y_train,y_train_pred)
  model_test_accuracy=accuracy_score(y_test,y_test_pred)
  accuracy[list(models)[i]+'_train']=model_train_accuracy
  accuracy[list(models)[i]+'_test']=model_test_accuracy
  print(f"For {list(models)[i]} : ")
  print(f"Training Test: {classification_report(y_train,y_train_pred)}")
  print(f"Testing Test: {classification_report(y_test,y_test_pred)}\n")

For Logistic Regression : 
Training Test:               precision    recall  f1-score   support

           0       0.94      0.92      0.93     19759
           1       0.92      0.94      0.93     19906

    accuracy                           0.93     39665
   macro avg       0.93      0.93      0.93     39665
weighted avg       0.93      0.93      0.93     39665

Testing Test:               precision    recall  f1-score   support

           0       0.90      0.87      0.89      4939
           1       0.88      0.90      0.89      4978

    accuracy                           0.89      9917
   macro avg       0.89      0.89      0.89      9917
weighted avg       0.89      0.89      0.89      9917


For Bernoulli Naive Bayes : 
Training Test:               precision    recall  f1-score   support

           0       0.90      0.93      0.92     19759
           1       0.93      0.89      0.91     19906

    accuracy                           0.91     39665
   macro avg       0.91    

In [26]:
pd.DataFrame(list(accuracy.values()),index=list(accuracy),columns=['Accuracy'])

,Accuracy
Logistic Regression_train,0.928955
Logistic Regression_test,0.887567
Bernoulli Naive Bayes_train,0.914030
Bernoulli Naive Bayes_test,0.853181


## We can infer that Logistic regression is performing quite well.

In [27]:
model=LogisticRegression()
model.fit(x_train,y_train)
pred_train=model.predict(x_train)
pred_test=model.predict(x_test)

In [28]:
print(pred_test)

[0 1 1 ... 1 0 0]


In [29]:
from sklearn.metrics import confusion_matrix
confusion_matrix(y_train,pred_train)

array([[18168,  1591],
       [ 1227, 18679]])

In [30]:
confusion_matrix(y_test,pred_test)

array([[4310,  629],
       [ 486, 4492]])

In [31]:
from sklearn.pipeline import Pipeline
prediction_pipeline=Pipeline(steps=[
    ('transform',tfidf),
    ('model',model)
])
prediction_pipeline.fit(df['review'],y)

Pipeline(steps=[('transform', TfidfVectorizer()),
                ('model', LogisticRegression())])

In [32]:
def predict(text:str):
  text=cleaner(text)
  text=stopword_removal(text)
  text=stemmer(text)
  pred=prediction_pipeline.predict([text])
  if pred[0]==0:
    return "Negative"
  return "Positive"
predict('the world is full of badness')

'Negative'

In [33]:
predict('everyone is happy.')

'Positive'

In [34]:
predict('The stock market crashed today as investors suffered heavy losses.')

'Negative'

In [35]:
predict('A massive fire destroyed several houses and left hundreds of people homeless.')

'Negative'

In [36]:
import pickle

with open('prediction_pipeline.pkl', 'wb') as f:
    pickle.dump(prediction_pipeline, f)

In [37]:
import os
os.mkdir('sentiment_analysis')